# kvroute · Phase 3b: real traffic, scarce caches, repeated seeds

One notebook, several **blocks**. Run setup once per session, then the blocks you have time for (each block saves its own folder; download the zips as you go). Total for all blocks: ~7 GPU-hours, split over 2-3 sessions.

**Settings:** *Accelerator* = **GPU T4 x2**, **Internet on**. VisionArena-Chat shards (~2 GB each) are downloaded from Hugging Face inside this session; its terms forbid redistribution, so the zips below contain only results (latencies, token counts, aggregate statistics), never conversation text or images.

| block | what | ~time |
|---|---|---|
| 0 | load probe: round-robin at 4 rates (doubling) (pick the 'high' load from it) | 25 min |
| A | headline: VisionArena, full and 25% cache, 3 seeds | 3.3 h |
| B | cache sweep: 50% cache | 0.6 h |
| C | load sweep: low and mid load | 1.1 h |
| D | production prefix pattern: Mooncake, full and 25% cache | 1.1 h |
| E | 4 workers (2 per T4), full cache | 0.6 h |


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!python --version

In [ ]:
!pip install -q "vllm==0.32.0" 2>&1 | tail -3
!python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__)"

In [ ]:
BRANCH = 'phase-3b'
%cd /kaggle/working
!rm -rf kvsim && git clone -q -b {BRANCH} https://github.com/mansidhamne/kvsim.git
%cd /kaggle/working/kvsim
!git log --oneline -1
!pip install -q -e ".[router,traces]" 2>&1 | tail -2

## Block 0 · load probe (done on 2026-11-15; skip unless the setup changes)
Round-robin only, one seed, 4 doubling conversation-start rates. 'High' = highest rate where round-robin's TTFT p99 stays under ~5 s: **1.2 conversations/s**.

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload visionarena --va-shards 0,1 --va-max-rows 9000 \
    --policies round_robin --seeds 0 --loads 0.3,0.6,1.2,2.4 --duration 150 --warmup 30 --out /kaggle/working/p3b/probe

In [ ]:
# From block 0 (2026-11-15): round-robin TTFT p99 1.65 s @0.3, 2.43 s @0.6, 3.04 s @1.2 (goodput 0.93), 8.28 s @2.4 (goodput 0.26)
HIGH = 1.2
MID, LOW = 0.8, 0.4
print(HIGH, MID, LOW)

## Block A · headline (VisionArena, full and 25% cache, 3 seeds, rotated policy order)

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload visionarena --va-shards 0,1 --va-max-rows 9000 \
    --loads {HIGH} --caches full,x0.25 --seeds 0,1,2 --out /kaggle/working/p3b/A

## Block B · scarce cache: 1,638 blocks per GPU (~26k tokens), 2 seeds (~1.1 h)
Block A: real per-GPU cache demand peaked at 23-52k tokens (median 10-16k), so 25% (65k) and 50% were not scarce. 1,638 blocks is below the peak, so evictions are certain, but large enough that routing decides whose history survives. (5%/819 blocks skipped: running requests alone fill it.)

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload visionarena --va-shards 0,1 --va-max-rows 9000 \
    --loads {HIGH} --caches 1638 --seeds 0,1 --out /kaggle/working/p3b/B2

## Block C · load sweep (full cache)

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload visionarena --va-shards 0,1 --va-max-rows 9000 \
    --loads {LOW},{MID} --caches full --seeds 0 --out /kaggle/working/p3b/C

## Block D · production prefix pattern (Mooncake, text only), full cache, 2 seeds (~1.1 h)
Real-speed replay (~3.2 requests/s, ~780-token prompts, outputs capped at 64 tokens) of a 6-minute window; seed 1 = a window 15 min later. No sessions, so session affinity can't pin anything; requests share prefixes across users (reuse ceiling ~0.22). The trace's live working set (~700k tokens) is far above the full cache (~200k per GPU), so this is already cache-scarce.

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload mooncake --loads 1 --caches full --seeds 0,1 --out /kaggle/working/p3b/D

## Block E · 4 workers (2 vLLM servers per T4). If startup fails with out-of-memory, skip it and say so.

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 4 --gpus 2 --workload visionarena --va-shards 0,1 --va-max-rows 9000 \
    --loads {HIGH} --caches full --seeds 0 --out /kaggle/working/p3b/E

## Block V · router v2 (`cache_aware_v2`: decode-aware estimate + switching cost)
Same setups as B2 and D, with v2 next to v1, session affinity and round-robin, and more seeds so the small v1-vs-affinity differences get firm. Policy order rotates with the seed. Two cells; they can run in separate sessions (run setup and the `HIGH` cell first each time).

### V1 · VisionArena, scarce cache (1,638 blocks/GPU), seeds 0-3, own replies (~2.4 h)

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload visionarena --va-shards 0,1 --va-max-rows 9000 \
    --loads {HIGH} --caches 1638 --seeds 0,1,2,3 --policies round_robin,session_affinity,cache_aware,cache_aware_v2 --out /kaggle/working/p3b/V1

### V2 · Mooncake, real speed, full cache, windows 0-2 (~1.6 h)

In [ ]:
!python -m deploy.kaggle_run phase3b --workers 2 --gpus 2 --workload mooncake --loads 1 --caches full \
    --seeds 0,1,2 --policies round_robin,session_affinity,cache_aware,cache_aware_v2 --out /kaggle/working/p3b/V2

## Save results (run after any block)
Expected size: tens of MB. If it's GBs, the dataset folder got included: don't download it.

In [ ]:
# Zips every block folder EXCEPT the ~4 GB VisionArena download (va_data).
!cd /kaggle/working/p3b && rm -f ../p3b_results.zip && zip -qr ../p3b_results.zip $(ls | grep -v '^va_data$') && ls -lh ../p3b_results.zip